# 05r - Temporal-representation reassessment

Diagnostica congelata dello stato iniziale: localizza il segnale per categoria canonica e ne misura la precisione temporale tramite shift teacher registrati. Non addestra né seleziona modelli.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Artefatti esatti e dataset logico

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.temporal_representation_reassessment import EXPECTED_05Q_INDEX_SHA256
from src.hayflow_model.temporal_observability_reassessment import EXPECTED_05P_INDEX_SHA256
from src.hayflow_model.axial_rich_state_recurrent_canary import EXPECTED_05O_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def exact_source(env_name,digest,message):
 override=os.environ.get(env_name);source=discover_indexed_artifact_source(INPUT_ROOT,digest,override=Path(override) if override else None);assert source is not None,message;return source
ARTIFACT_05Q_SOURCE=exact_source('HAYFLOW_05Q_ARTIFACT',EXPECTED_05Q_INDEX_SHA256,'Artefatto 05q esatto non trovato.')
ARTIFACT_05P_SOURCE=exact_source('HAYFLOW_05P_ARTIFACT',EXPECTED_05P_INDEX_SHA256,'Artefatto 05p esatto non trovato.')
ARTIFACT_05O_SOURCE=exact_source('HAYFLOW_05O_ARTIFACT',EXPECTED_05O_INDEX_SHA256,'Artefatto 05o esatto non trovato.')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_size';stamp=str(source.stat().st_size)
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow05r_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05q':str(ARTIFACT_05Q_SOURCE),'05p':str(ARTIFACT_05P_SOURCE),'05o':str(ARTIFACT_05O_SOURCE),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 05r][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880 and not bundle.manifest['physical_merge_performed'];print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Decomposizione e shift temporali congelati

Il preflight verifica l’intera catena 05o→05p→05q. L’esecuzione stampa tre sole righe, una per checkpoint joint, e non modifica i pesi.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import TemporalRepresentationReassessment,TemporalRepresentationReassessmentConfig
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_temporal_representation_reassessment.yml').read_text());config=TemporalRepresentationReassessmentConfig.from_mapping(cfg['temporal_representation_reassessment'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_temporal_representation_reassessment');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=TemporalRepresentationReassessment(bundle,OUTPUT_DIR,config,ARTIFACT_05Q_SOURCE,ARTIFACT_05P_SOURCE,ARTIFACT_05O_SOURCE,code_revision=REVISION);preflight=session.prepare();display({'valid':preflight['valid'],'categories':preflight['active_state_categories'],'frozen_joint_only':preflight['frozen_joint_checkpoint_only'],'future_state_diagnostic_only':preflight['future_teacher_state_used_only_for_registered_time_shift_diagnostic']});assert preflight['valid'] and not preflight['retraining_performed']

In [ ]:
try:
 final_report=session.run()
finally:
 session.close()
def compact(row):return {'median':round(row['median_rmse_gain_fraction'],4),'regenerative':round(row['median_regenerative_gain_fraction'],4),'wins':row['positive_win_count']}
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'checkpoint_error_mv':final_report['maximum_checkpoint_reproduction_error_mv'],'sketch_error':final_report['maximum_full_sketch_reconstruction_error'],'category_ablations':{name:compact(row) for name,row in final_report['state_category_ablation_degradations'].items()},'time_shifts':{name:compact(row) for name,row in final_report['teacher_state_time_shift_degradations'].items()},'material_categories':final_report['material_state_categories'],'shift_signals':final_report['teacher_state_time_shift_signals'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['retraining_performed'] and not final_report['model_or_training_authorized']

## 3. Crea e scarica lo ZIP con il downloader browser stabile

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_temporal_representation_reassessment','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})